# exp5 R2 2 complexity latency

**Self-contained Colab notebook.** Run the setup cells once (top to bottom), then the experiment. The four shared modules are written to `/content` automatically, so you do NOT need to upload a `shared/` folder. Edit Drive paths only in the `config_paths.py` cell below.

> Runtime: set **Runtime -> Change runtime type -> GPU** for the inference and training experiments.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
print('Drive mounted; /content on path.')

In [ ]:
# scikit-image backs metrics.calc_ssim; guarantee it for every notebook
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-image'])
print('scikit-image ready')

In [ ]:
%%writefile /content/config_paths.py
"""
Central path + constant configuration for all JBHI revision Colab experiments.

Derived verbatim from the author's existing notebooks:
  - COLAB_IEEE_FIGURES.ipynb      (paper baselines + EPHNOGRAM chunks + split)
  - cross_dataset_validation.ipynb (SensSmart-Tech + Trail-4 generator)

Every experiment script imports from this file so paths live in ONE place.
If your Drive layout differs, edit ONLY this file.
"""

import os
import torch

# ============================================================================
# >>> EDIT THIS BLOCK IF YOUR DRIVE MOVES <<<
# Updated 2026-07-14: the dataset is now a SHARED folder mounted at
# /content/drive/MyDrive/JBHI-Revision/October Thesis Dataset. If it moves again,
# change ONLY DRIVE_ROOT and OCT_DIR. NOTE: DATA is READ from OCT_DIR; everything
# the code WRITES (retrained checkpoints, outputs) goes under your own Drive
# (OUT_ROOT), so a read-only shared dataset still works.
# ============================================================================
DRIVE_ROOT = "/content/drive/MyDrive"
# Shared dataset root: contains "Refine-Chunked", "Best-epoch-trail-4", etc. (read-only OK).
OCT_DIR    = f"{DRIVE_ROOT}/JBHI-Revision/October Thesis Dataset"

# ----------------------------------------------------------------------------
# EPHNOGRAM data (training-domain test set = 453 segments, 3 subjects)
# .npz keys per chunk: 'ecg', 'pcg', 'subject_id', 'scenario'
#   ecg, pcg : float32 arrays, 64000 samples (8 s @ 8 kHz), RAW (not normalized)
#   subject_id : e.g. 'S009', 'S019', 'S023'
#   scenario   : free-text, e.g. 'rest', 'exercise bicycle stress test'
# ----------------------------------------------------------------------------
EPH_CHUNKS_DIR = f"{OCT_DIR}/Refine-Chunked"
# Subject split. keys: train/val/test_subjects. IF THIS FILE IS ABSENT the
# notebooks fall back to an arbitrary split, which gives the WRONG test set
# (the 2026-07-10 run hit this: 22 segments / 4 subjects instead of the paper's
# 453 segments / 3 subjects). Upload the real split JSON, or point this at it.
SPLIT_FILE     = f"{EPH_CHUNKS_DIR}/train_val_test_split.json"

# ----------------------------------------------------------------------------
# Model checkpoints  --  CONFIRM ALL FOUR EXIST IN THE NEW ACCOUNT
#
#   "Proposed" MUST be the Trail-4 generator (polarity fix, BatchNorm output),
#   i.e. trail4_best.pt or trail4_generator.pth. A "trial2_*" / Trail-2 file is
#   the WRONG (Tanh) model: it loads only partially under strict=False and
#   produces a near-flat, invalid reconstruction (see the 2026-07-10 run).
#
#   The three baselines lived under a SEPARATE top-level folder
#   (Paper1_ECG_Reconstruction) in the old account. Confirm they were migrated;
#   if they now sit under OCT_DIR, repoint them below.
# ----------------------------------------------------------------------------
CKPT = {
    # Baselines are RETRAINED by exp7 and SAVED to your own writable Drive
    # (DRIVE_ROOT/JBHI_Revision/baseline_checkpoints == OUT_ROOT/baseline_checkpoints),
    # so a read-only shared dataset is fine. Same paths are used to load them later.
    "CNN":      f"{DRIVE_ROOT}/JBHI_Revision/baseline_checkpoints/CNN_EncoderDecoder/best_model.pt",
    "UNet":     f"{DRIVE_ROOT}/JBHI_Revision/baseline_checkpoints/UNet_Baseline/best_model.pt",
    "WGAN-GP":  f"{DRIVE_ROOT}/JBHI_Revision/baseline_checkpoints/WGAN_GP/best_model.pt",
    "Proposed": f"{OCT_DIR}/Best-epoch-trail-4/trail4_best.pt",   # read from the shared dataset
}
CKPT_PROPOSED_ALT = f"{OCT_DIR}/trail4_generator.pth"   # fallback copy (state_dict)

# ----------------------------------------------------------------------------
# SensSmart-Tech cross-dataset (32 subjects, 338 records @ 1 kHz -> 8 kHz)
#   WFDB pairs: {base}_ecg.hea/.dat and {base}_pcg.hea/.dat ; ECG lead II
# ----------------------------------------------------------------------------
SENSSMART_WFDB  = f"{OCT_DIR}/senssmarttech-database/WFDB"
SENSSMART_PREP  = f"{OCT_DIR}/milestone2_preprocessed"     # cached .npz (keys ecg/pcg)

# ----------------------------------------------------------------------------
# Output root for ALL revision artifacts (created on demand)
#   Each experiment writes to a named sub-folder here.
# ----------------------------------------------------------------------------
OUT_ROOT = f"{DRIVE_ROOT}/JBHI_Revision"

def out_dir(name):
    """Return (and create) an output sub-folder under OUT_ROOT."""
    p = os.path.join(OUT_ROOT, name)
    os.makedirs(p, exist_ok=True)
    return p

# ----------------------------------------------------------------------------
# Signal + inference constants
# ----------------------------------------------------------------------------
FS          = 8000        # sampling rate (Hz)
SEG_LEN     = 64000       # samples per segment (8 s)
NUM_SCENARIOS = 3         # rest / exercise / recovery one-hot
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# R-peak detector settings (identical to metrics.ipynb; keep fixed for reproducibility)
RPEAK_DISTANCE   = 400    # min inter-peak samples = 50 ms @ 8 kHz
RPEAK_PROMINENCE = 0.5

# SensSmart preprocessing bands (from cross_dataset_validation.ipynb)
ECG_BAND = (0.5, 40.0)
PCG_BAND = (25.0, 400.0)
FILTER_ORDER = 4

# Scenario one-hots
SCEN_REST     = [1.0, 0.0, 0.0]
SCEN_EXERCISE = [0.0, 1.0, 0.0]
SCEN_RECOVERY = [0.0, 0.0, 1.0]


def scenario_to_index(scenario_str):
    """Map EPHNOGRAM free-text scenario -> class index {0,1,2}.

    Matches the mapping used in COLAB_IEEE_FIGURES.ipynb.
    """
    s = str(scenario_str).lower()
    if "rest" in s or "laying" in s:
        return 0
    if "exercise" in s or "bicycle" in s or "treadmill" in s or "running" in s:
        return 1
    return 2

In [ ]:
%%writefile /content/models.py
"""
Model architectures for the JBHI revision experiments.

Reproduced VERBATIM from the author's COLAB_IEEE_FIGURES.ipynb (paper Table II
baselines) and models.ipynb (ablation trails), so the checkpoints in
config_paths.CKPT load cleanly.

Which class loads which checkpoint
----------------------------------
    CKPT['CNN']      -> CNNEncoderDecoder()        # no scenario, forward(x)
    CKPT['UNet']     -> UNetGenerator()            # pix2pix, forward(x)
    CKPT['WGAN-GP']  -> UNetGenerator()            # same generator as U-Net
    CKPT['Proposed'] -> ImprovedGeneratorNoTanh()  # scenario, forward(x, s_onehot)

The proposed model's forward takes a scenario one-hot; the three baselines take
only the PCG. Every inference script must branch on this (helper `run_generator`
below does it for you).

For the R1.4 component ablation, use `ConfigurableGenerator`, which exposes the
three grouped factors as switches: use_attention, use_dilated, and
output_activation in {'batchnorm','tanh','linear'}.

NOTE ON PARAM COUNTS: the submitted paper's numbers (CNN 9.8M, U-Net 13.6M,
Proposed 14.0M) do NOT match these classes (which give ~larger counts). Always
report count_params(model_loaded_from_checkpoint) computed here -- see R2.2/R2.5.
"""

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ===========================================================================
# Shared bottleneck modules
# ===========================================================================
class MultiHeadSelfAttention(nn.Module):
    """8-head self-attention over the temporal axis at the bottleneck."""
    def __init__(self, channels, num_heads=8, dropout=0.1):
        super().__init__()
        assert channels % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Conv1d(channels, channels * 3, kernel_size=1, bias=False)
        self.proj = nn.Conv1d(channels, channels, kernel_size=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, C, T = x.shape
        qkv = self.qkv(x).reshape(B, 3, self.num_heads, self.head_dim, T)
        qkv = qkv.permute(1, 0, 2, 4, 3)          # (3, B, heads, T, head_dim)
        # .contiguous() is REQUIRED for SDPA to select the flash / memory-efficient
        # kernel (O(T) memory). Without it the permuted views fall back to the MATH
        # kernel, which materializes the T x T score matrix -- that OOM-ed the
        # attention variants even on a 40 GB A100 at batch 48. Mathematically
        # identical to softmax(QK^T/sqrt(d))V, so existing checkpoints are unaffected.
        q, k, v = qkv[0].contiguous(), qkv[1].contiguous(), qkv[2].contiguous()
        p = self.dropout.p if self.training else 0.0
        out = F.scaled_dot_product_attention(q, k, v, dropout_p=p)
        out = out.transpose(2, 3).reshape(B, C, T)
        return self.proj(out) + x


class MultiScaleDilatedBlock(nn.Module):
    """Parallel dilated convs at rates 1, 2, 4 with a 1x1 fusion + residual."""
    def __init__(self, channels):
        super().__init__()
        c1 = channels // 3 + (1 if channels % 3 >= 1 else 0)
        c2 = channels // 3 + (1 if channels % 3 >= 2 else 0)
        c3 = channels // 3
        self.conv_d1 = nn.Conv1d(channels, c1, 3, padding=1, dilation=1)
        self.conv_d2 = nn.Conv1d(channels, c2, 3, padding=2, dilation=2)
        self.conv_d4 = nn.Conv1d(channels, c3, 3, padding=4, dilation=4)
        self.fusion = nn.Sequential(
            nn.Conv1d(channels, channels, 1),
            nn.BatchNorm1d(channels), nn.ReLU(inplace=True))

    def forward(self, x):
        cat = torch.cat([self.conv_d1(x), self.conv_d2(x), self.conv_d4(x)], dim=1)
        return self.fusion(cat) + x


class EnhancedResidualBlock(nn.Module):
    """Residual block with optional MHSA and MSDC (Trails 2-4)."""
    def __init__(self, channels, use_attention=False, use_dilated=True):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm1d(channels)
        self.use_dilated = use_dilated
        if use_dilated:
            self.dilated_block = MultiScaleDilatedBlock(channels)
        self.conv2 = nn.Conv1d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(channels)
        self.use_attention = use_attention
        if use_attention:
            self.attention = MultiHeadSelfAttention(channels, num_heads=8)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        if self.use_dilated:
            out = self.dilated_block(out)
        out = self.bn2(self.conv2(out))
        if self.use_attention:
            out = self.attention(out)
        return self.relu(out + residual)


# ===========================================================================
# Paper Table II baseline: CNN Encoder-Decoder (8-level, no skips, no scenario)
# ===========================================================================
class ConvBlock(nn.Module):
    def __init__(self, i, o, k=4, s=2, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(i, o, k, s, p, bias=False),
            nn.BatchNorm1d(o), nn.LeakyReLU(0.2, inplace=True))
    def forward(self, x): return self.block(x)


class DeconvBlock(nn.Module):
    def __init__(self, i, o, k=4, s=2, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose1d(i, o, k, s, p, bias=False),
            nn.BatchNorm1d(o), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)


class CNNEncoderDecoder(nn.Module):
    """Paper CNN baseline -> loads CKPT['CNN']. forward(x)."""
    def __init__(self, in_channels=1, out_channels=1, base_filters=64):
        super().__init__()
        bf = base_filters
        self.enc1 = ConvBlock(in_channels, bf)
        self.enc2 = ConvBlock(bf, bf * 2)
        self.enc3 = ConvBlock(bf * 2, bf * 4)
        self.enc4 = ConvBlock(bf * 4, bf * 8)
        self.enc5 = ConvBlock(bf * 8, bf * 8)
        self.enc6 = ConvBlock(bf * 8, bf * 8)
        self.enc7 = ConvBlock(bf * 8, bf * 8)
        self.enc8 = ConvBlock(bf * 8, bf * 8)
        self.dec1 = DeconvBlock(bf * 8, bf * 8)
        self.dec2 = DeconvBlock(bf * 8, bf * 8)
        self.dec3 = DeconvBlock(bf * 8, bf * 8)
        self.dec4 = DeconvBlock(bf * 8, bf * 8)
        self.dec5 = DeconvBlock(bf * 8, bf * 4)
        self.dec6 = DeconvBlock(bf * 4, bf * 2)
        self.dec7 = DeconvBlock(bf * 2, bf)
        self.final = nn.ConvTranspose1d(bf, out_channels, 4, 2, 1)

    def forward(self, x):
        n = x.size(2)
        e = self.enc1(x); e = self.enc2(e); e = self.enc3(e); e = self.enc4(e)
        e = self.enc5(e); e = self.enc6(e); e = self.enc7(e); e = self.enc8(e)
        d = self.dec1(e); d = self.dec2(d); d = self.dec3(d); d = self.dec4(d)
        d = self.dec5(d); d = self.dec6(d); d = self.dec7(d)
        out = self.final(d)
        if out.size(2) != n:
            diff = n - out.size(2)
            out = F.pad(out, (diff // 2, diff - diff // 2))
        return out


# ===========================================================================
# Paper Table II baseline: pix2pix U-Net (shared by U-Net and WGAN-GP)
# ===========================================================================
class UNetBlock(nn.Module):
    def __init__(self, i, o, normalize=True, dropout=0.0):
        super().__init__()
        layers = [nn.Conv1d(i, o, 4, 2, 1, bias=False)]
        if normalize:
            layers.append(nn.BatchNorm1d(o))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)


class UNetUpBlock(nn.Module):
    def __init__(self, i, o, dropout=0.0):
        super().__init__()
        layers = [nn.ConvTranspose1d(i, o, 4, 2, 1, bias=False),
                  nn.BatchNorm1d(o), nn.ReLU(inplace=True)]
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x, skip):
        x = self.block(x)
        if x.size(2) != skip.size(2):
            diff = skip.size(2) - x.size(2)
            x = F.pad(x, (diff // 2, diff - diff // 2))
        return torch.cat([x, skip], dim=1)


class UNetGenerator(nn.Module):
    """Paper U-Net / WGAN-GP generator -> loads CKPT['UNet'] or CKPT['WGAN-GP']. forward(x)."""
    def __init__(self, in_channels=1, out_channels=1, features=64):
        super().__init__()
        f = features
        self.e1 = UNetBlock(in_channels, f, normalize=False)
        self.e2 = UNetBlock(f, f * 2); self.e3 = UNetBlock(f * 2, f * 4)
        self.e4 = UNetBlock(f * 4, f * 8); self.e5 = UNetBlock(f * 8, f * 8)
        self.e6 = UNetBlock(f * 8, f * 8); self.e7 = UNetBlock(f * 8, f * 8)
        self.e8 = UNetBlock(f * 8, f * 8, normalize=False)
        self.d1 = UNetUpBlock(f * 8, f * 8, dropout=0.5)
        self.d2 = UNetUpBlock(f * 16, f * 8, dropout=0.5)
        self.d3 = UNetUpBlock(f * 16, f * 8, dropout=0.5)
        self.d4 = UNetUpBlock(f * 16, f * 8)
        self.d5 = UNetUpBlock(f * 16, f * 4); self.d6 = UNetUpBlock(f * 8, f * 2)
        self.d7 = UNetUpBlock(f * 4, f)
        self.final = nn.ConvTranspose1d(f * 2, out_channels, 4, 2, 1)

    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(e1); e3 = self.e3(e2); e4 = self.e4(e3)
        e5 = self.e5(e4); e6 = self.e6(e5); e7 = self.e7(e6); e8 = self.e8(e7)
        d = self.d1(e8, e7); d = self.d2(d, e6); d = self.d3(d, e5); d = self.d4(d, e4)
        d = self.d5(d, e3); d = self.d6(d, e2); d = self.d7(d, e1)
        out = self.final(d)
        if out.size(2) != x.size(2):
            diff = x.size(2) - out.size(2)
            out = F.pad(out, (diff // 2, diff - diff // 2))
        return out


# ===========================================================================
# Proposed generator (Trail-4): U-Net + MHSA + MSDC + scenario, BatchNorm output
# ===========================================================================
class ImprovedGeneratorNoTanh(nn.Module):
    """Proposed model -> loads CKPT['Proposed']. forward(x, scenario_onehot)."""
    def __init__(self, signal_length=64000, base_channels=64, num_scenarios=3):
        super().__init__()
        bc = base_channels
        self.scenario_embed = nn.Linear(num_scenarios, bc)
        self.enc1 = nn.Sequential(nn.Conv1d(1, bc, 7, 2, 3), nn.BatchNorm1d(bc), nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv1d(bc, bc*2, 5, 2, 2), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.enc3 = nn.Sequential(nn.Conv1d(bc*2, bc*4, 5, 2, 2), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.enc4 = nn.Sequential(nn.Conv1d(bc*4, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.enc5 = nn.Sequential(nn.Conv1d(bc*8, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.bottleneck = nn.Sequential(
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True))
        self.dec5 = nn.Sequential(nn.ConvTranspose1d(bc*8, bc*8, 4, 2, 1), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.dec4 = nn.Sequential(nn.Conv1d(bc*16, bc*8, 1), nn.ConvTranspose1d(bc*8, bc*4, 4, 2, 1), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.dec3 = nn.Sequential(nn.Conv1d(bc*8, bc*4, 1), nn.ConvTranspose1d(bc*4, bc*2, 4, 2, 1), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.dec2 = nn.Sequential(nn.Conv1d(bc*4, bc*2, 1), nn.ConvTranspose1d(bc*2, bc, 4, 2, 1), nn.BatchNorm1d(bc), nn.ReLU(True))
        self.dec1 = nn.Sequential(nn.Conv1d(bc*2, bc, 1), nn.ConvTranspose1d(bc, 1, 4, 2, 1), nn.BatchNorm1d(1))

    def forward(self, x, scenario_onehot):
        s = self.scenario_embed(scenario_onehot).unsqueeze(2)
        e1 = self.enc1(x) + s
        e2 = self.enc2(e1); e3 = self.enc3(e2); e4 = self.enc4(e3); e5 = self.enc5(e4)
        b = self.bottleneck(e5)
        d5 = torch.cat([self.dec5(b), e4], dim=1)
        d4 = torch.cat([self.dec4(d5), e3], dim=1)
        d3 = torch.cat([self.dec3(d4), e2], dim=1)
        d2 = torch.cat([self.dec2(d3), e1], dim=1)
        return self.dec1(d2)


# ===========================================================================
# Configurable generator for the R1.4 component ablation
# ===========================================================================
class ConfigurableGenerator(nn.Module):
    """Ablation generator: toggle each grouped factor independently.

    use_attention : add MHSA in the 4 bottleneck blocks
    use_dilated   : add MSDC in the 4 bottleneck blocks
    output_activation : 'batchnorm' (proposed) | 'tanh' | 'linear'
    use_scenario  : condition on the scenario one-hot (keep True to match proposed)

    Ablation rows the reviewer asked for map to:
        U-Net only          -> use_attention=False, use_dilated=False
        U-Net + MSDC        -> use_attention=False, use_dilated=True
        U-Net + MHSA        -> use_attention=True,  use_dilated=False
        U-Net + both        -> use_attention=True,  use_dilated=True
        Proposed + Tanh     -> both True, output_activation='tanh'
        Proposed + linear   -> both True, output_activation='linear'
        Proposed (BatchNorm)-> both True, output_activation='batchnorm'
    Polarity-on/off is a DATA-loader switch; adversarial-on/off is a TRAINING
    switch -- neither is an architecture change, so control them in the trainer.
    """
    def __init__(self, base_channels=64, num_scenarios=3,
                 use_attention=True, use_dilated=True,
                 output_activation="batchnorm", use_scenario=True):
        super().__init__()
        bc = base_channels
        self.use_scenario = use_scenario
        if use_scenario:
            self.scenario_embed = nn.Linear(num_scenarios, bc)
        self.enc1 = nn.Sequential(nn.Conv1d(1, bc, 7, 2, 3), nn.BatchNorm1d(bc), nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv1d(bc, bc*2, 5, 2, 2), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.enc3 = nn.Sequential(nn.Conv1d(bc*2, bc*4, 5, 2, 2), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.enc4 = nn.Sequential(nn.Conv1d(bc*4, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.enc5 = nn.Sequential(nn.Conv1d(bc*8, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.bottleneck = nn.Sequential(*[
            EnhancedResidualBlock(bc*8, use_attention=use_attention, use_dilated=use_dilated)
            for _ in range(4)])
        self.dec5 = nn.Sequential(nn.ConvTranspose1d(bc*8, bc*8, 4, 2, 1), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.dec4 = nn.Sequential(nn.Conv1d(bc*16, bc*8, 1), nn.ConvTranspose1d(bc*8, bc*4, 4, 2, 1), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.dec3 = nn.Sequential(nn.Conv1d(bc*8, bc*4, 1), nn.ConvTranspose1d(bc*4, bc*2, 4, 2, 1), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.dec2 = nn.Sequential(nn.Conv1d(bc*4, bc*2, 1), nn.ConvTranspose1d(bc*2, bc, 4, 2, 1), nn.BatchNorm1d(bc), nn.ReLU(True))
        head = [nn.Conv1d(bc*2, bc, 1), nn.ConvTranspose1d(bc, 1, 4, 2, 1)]
        if output_activation == "batchnorm":
            head.append(nn.BatchNorm1d(1))
        elif output_activation == "tanh":
            head.append(nn.Tanh())
        elif output_activation == "linear":
            pass
        else:
            raise ValueError(f"unknown output_activation {output_activation}")
        self.dec1 = nn.Sequential(*head)

    def forward(self, x, scenario_onehot=None):
        e1 = self.enc1(x)
        if self.use_scenario and scenario_onehot is not None:
            e1 = e1 + self.scenario_embed(scenario_onehot).unsqueeze(2)
        e2 = self.enc2(e1); e3 = self.enc3(e2); e4 = self.enc4(e3); e5 = self.enc5(e4)
        b = self.bottleneck(e5)
        d5 = torch.cat([self.dec5(b), e4], dim=1)
        d4 = torch.cat([self.dec4(d5), e3], dim=1)
        d3 = torch.cat([self.dec3(d4), e2], dim=1)
        d2 = torch.cat([self.dec2(d3), e1], dim=1)
        return self.dec1(d2)


# ===========================================================================
# Discriminator (for R1.4 adversarial retraining)
# ===========================================================================
class SpectralNormDiscriminator(nn.Module):
    """Spectrally-normalized PatchGAN critic with scenario conditioning."""
    def __init__(self, base_channels=64, num_scenarios=3):
        super().__init__()
        bc = base_channels
        self.scenario_embed = nn.Linear(num_scenarios, bc)
        sn = nn.utils.spectral_norm
        self.conv1 = sn(nn.Conv1d(2, bc, 7, 2, 3))
        self.conv2 = sn(nn.Conv1d(bc, bc*2, 5, 2, 2)); self.bn2 = nn.BatchNorm1d(bc*2)
        self.conv3 = sn(nn.Conv1d(bc*2, bc*4, 5, 2, 2)); self.bn3 = nn.BatchNorm1d(bc*4)
        self.conv4 = sn(nn.Conv1d(bc*4, bc*8, 5, 2, 2)); self.bn4 = nn.BatchNorm1d(bc*8)
        self.conv5 = sn(nn.Conv1d(bc*8, bc*8, 5, 2, 2)); self.bn5 = nn.BatchNorm1d(bc*8)
        self.pool = nn.AdaptiveAvgPool1d(1); self.fc = nn.Linear(bc*8, 1)
        self.lrelu = nn.LeakyReLU(0.2, True)

    def forward(self, pcg, ecg, scenario_onehot):
        x = torch.cat([pcg, ecg], dim=1)
        s = self.scenario_embed(scenario_onehot).unsqueeze(2)
        x = self.lrelu(self.conv1(x)) + s
        x = self.lrelu(self.bn2(self.conv2(x)))
        x = self.lrelu(self.bn3(self.conv3(x)))
        x = self.lrelu(self.bn4(self.conv4(x)))
        x = self.lrelu(self.bn5(self.conv5(x)))
        return torch.sigmoid(self.fc(self.pool(x).squeeze(2)))


# ===========================================================================
# WGAN-GP critic (for retraining the WGAN-GP baseline; extracted verbatim from
# the original 05_GAN_WGAN_GP notebook). No BatchNorm, no sigmoid -> a valid
# WGAN critic. The generator is the plain UNetGenerator above.
# ===========================================================================
class WGANCritic(nn.Module):
    def __init__(self, in_channels=2, features=64):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv1d(in_channels, features, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features, features * 2, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features * 2, features * 4, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features * 4, features * 8, 4, 1, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features * 8, 1, 4, 1, 1),
        )

    def forward(self, x, y):
        return self.model(torch.cat([x, y], dim=1))


def compute_gradient_penalty(critic, pcg, real_ecg, fake_ecg, device):
    """WGAN-GP gradient penalty. Interpolates the ECG only; PCG conditions."""
    bs = real_ecg.size(0)
    alpha = torch.rand(bs, 1, 1, device=device)
    interp = (alpha * real_ecg + (1 - alpha) * fake_ecg).requires_grad_(True)
    c_interp = critic(pcg, interp)
    grads = torch.autograd.grad(
        outputs=c_interp, inputs=interp,
        grad_outputs=torch.ones_like(c_interp),
        create_graph=True, retain_graph=True)[0]
    grads = grads.view(bs, -1)
    return ((grads.norm(2, dim=1) - 1) ** 2).mean()


# ===========================================================================
# Loading + utility helpers
# ===========================================================================
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def load_model(model, checkpoint_path, device="cpu", strict=False):
    """Load a checkpoint into `model`, handling common save formats."""
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    if isinstance(ckpt, dict):
        for key in ("generator_state_dict", "model_state_dict", "state_dict"):
            if key in ckpt:
                ckpt = ckpt[key]; break
    state = {k.replace("module.", ""): v for k, v in ckpt.items()} \
            if isinstance(ckpt, dict) else ckpt
    result = model.load_state_dict(state, strict=strict)
    # Loud guard: a large key mismatch means the checkpoint does not fit this
    # architecture (e.g. a Trail-2/trial2 file loaded into the Trail-4 model).
    n_model = max(len(model.state_dict()), 1)
    n_missing, n_unexpected = len(result.missing_keys), len(result.unexpected_keys)
    tag = os.path.basename(checkpoint_path)
    if n_missing or n_unexpected:
        flag = "  <-- LIKELY WRONG CHECKPOINT FOR THIS MODEL" if n_missing > 0.10 * n_model else ""
        print(f"[load_model] {tag}: {n_missing}/{n_model} params missing, "
              f"{n_unexpected} unexpected{flag}")
    else:
        print(f"[load_model] {tag}: exact match ({n_model} tensors).")
    model.to(device).eval()
    return model, result


def build_paper_models(ckpt_dict, device="cpu"):
    """Load all four paper models. Returns {name: model}."""
    models = {}
    models["CNN"], _     = load_model(CNNEncoderDecoder(), ckpt_dict["CNN"], device)
    models["UNet"], _    = load_model(UNetGenerator(), ckpt_dict["UNet"], device)
    models["WGAN-GP"], _ = load_model(UNetGenerator(), ckpt_dict["WGAN-GP"], device)
    models["Proposed"], _= load_model(ImprovedGeneratorNoTanh(), ckpt_dict["Proposed"], device)
    return models


def run_generator(model, pcg_tensor, scenario_tensor, is_proposed):
    """Uniform inference call. `is_proposed` selects the (x, s) signature."""
    with torch.no_grad():
        if is_proposed:
            out = model(pcg_tensor, scenario_tensor)
        else:
            out = model(pcg_tensor)
    return out.squeeze().cpu().numpy()

In [ ]:
%%writefile /content/metrics.py
"""
Metrics for the JBHI revision.

Two families:
  (A) Signal-fidelity metrics  -- identical to the submitted paper's metrics.ipynb
                                  (PSNR, SNR, SSIM, Pearson r, MAE, RMSE,
                                   R-peak AMPLITUDE error, QRS-width error).
  (B) Temporal R-peak DETECTION metrics  -- NEW, added to answer Reviewer 1 #1.
       These detect R-peaks INDEPENDENTLY in the reconstructed ECG and match
       them to ground-truth peaks within a tolerance window, then report
       sensitivity, PPV, F1, FP, FN, timing-error stats, RR-interval error,
       and heart-rate error. This is the metric the paper was missing.

Keep the detector settings fixed (config_paths.RPEAK_*) so every experiment and
the paper agree.
"""

import numpy as np
import scipy.signal as signal
from scipy.stats import pearsonr


# ---------------------------------------------------------------------------
# R-peak detection (single, shared definition)
# ---------------------------------------------------------------------------
def detect_r_peaks(ecg, fs=8000, distance=400, prominence=0.5):
    """scipy find_peaks on an ECG trace. Returns integer sample indices."""
    peaks, _ = signal.find_peaks(np.asarray(ecg).ravel(),
                                 distance=distance, prominence=prominence)
    return peaks


# ---------------------------------------------------------------------------
# (A) Signal-fidelity metrics
# ---------------------------------------------------------------------------
def calc_psnr(real, fake):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    mse = np.mean((real - fake) ** 2)
    if mse < 1e-10:
        return 100.0
    peak = np.max(np.abs(real))
    return float(20 * np.log10(peak / np.sqrt(mse)))


def calc_snr(real, fake):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    noise = real - fake
    return float(10 * np.log10(np.sum(real ** 2) / (np.sum(noise ** 2) + 1e-12)))


def calc_correlation(real, fake):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    try:
        c, _ = pearsonr(real, fake)
        return float(c)
    except Exception:
        return 0.0


def calc_ssim(real, fake):
    from skimage.metrics import structural_similarity as compare_ssim  # lazy: Colab-only dep
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    r = (real - real.min()) / (real.max() - real.min() + 1e-8)
    f = (fake - fake.min()) / (fake.max() - fake.min() + 1e-8)
    return float(compare_ssim(r, f, data_range=1.0))


def calc_mae(real, fake):
    return float(np.mean(np.abs(np.asarray(real).ravel() - np.asarray(fake).ravel())))


def calc_rmse(real, fake):
    d = np.asarray(real).ravel() - np.asarray(fake).ravel()
    return float(np.sqrt(np.mean(d ** 2)))


def calc_r_peak_amplitude_error(real, fake, fs=8000, normalize01=True, **kw):
    """Paper's 'R-peak error' -> renamed 'R-peak AMPLITUDE error'.

    Detects R-peaks on the GROUND TRUTH only, then averages the amplitude error at
    those locations. Kept ONLY as a fidelity measure; NOT a timing metric (R1 #1).

    normalize01=True (DEFAULT) min-max normalizes both signals to [0,1] before the
    error, which REPRODUCES the paper's ~0.126 scale (confirmed: the paper's metric
    is [0,1]-normalized). normalize01=False gives the raw z-scored form, which is
    ~8x larger (z-scored ECG spans ~6-8 units vs 1 for [0,1]) -- that is what the
    first exp4/6/7 runs reported (~1.0). Use the default for consistency with the
    manuscript. Returns np.nan when the ground truth has no detectable R-peaks.
    """
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    peaks = detect_r_peaks(real, fs=fs, **kw)
    if len(peaks) == 0:
        return np.nan
    if normalize01:
        real = (real - real.min()) / (real.max() - real.min() + 1e-8)
        fake = (fake - fake.min()) / (fake.max() - fake.min() + 1e-8)
    return float(np.mean(np.abs(real[peaks] - fake[peaks])))


def _qrs_width_at_peak(ecg, peak_idx, fs=8000, threshold=0.5, search_ms=25):
    amp = abs(ecg[peak_idx]); thr = threshold * amp
    win = int(search_ms / 1000 * fs)
    q = peak_idx
    for j in range(peak_idx - 1, max(peak_idx - win, 0), -1):
        if abs(ecg[j]) < thr:
            q = j; break
    s = peak_idx
    for j in range(peak_idx + 1, min(peak_idx + win, len(ecg))):
        if abs(ecg[j]) < thr:
            s = j; break
    return (s - q) / fs * 1000.0


def calc_qrs_width_error(real, fake, fs=8000, **kw):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    peaks = detect_r_peaks(real, fs=fs, **kw)
    if len(peaks) == 0:
        return np.nan
    errs = [abs(_qrs_width_at_peak(real, p, fs) - _qrs_width_at_peak(fake, p, fs))
            for p in peaks]
    return float(np.mean(errs))


# ---------------------------------------------------------------------------
# (B) Temporal R-peak DETECTION metrics  (NEW -- Reviewer 1 #1)
# ---------------------------------------------------------------------------
def match_peaks(true_peaks, pred_peaks, tol_samples):
    """Greedy nearest-neighbour matching within a tolerance window.

    Each true peak claims the closest unused predicted peak within tol_samples.
    Returns (matches, TP, FP, FN, signed_timing_errors_samples).
    signed error = pred_index - true_index (positive = reconstructed peak late).
    """
    true_peaks = list(np.asarray(true_peaks).astype(int))
    pred_peaks = list(np.asarray(pred_peaks).astype(int))
    used, matches, terr = set(), [], []
    for tp in true_peaks:
        cands = [(abs(pp - tp), pp) for pp in pred_peaks
                 if pp not in used and abs(pp - tp) <= tol_samples]
        if cands:
            _, pp = min(cands)
            used.add(pp); matches.append((tp, pp)); terr.append(pp - tp)
    TP = len(matches)
    FN = len(true_peaks) - TP
    FP = len(pred_peaks) - TP
    return matches, TP, FP, FN, np.asarray(terr, dtype=float)


def temporal_rpeak_metrics(true_ecg, pred_ecg, fs=8000, tol_ms=50,
                           distance=400, prominence=0.5):
    """Full temporal detection report for ONE segment.

    Returns a dict with detection (Se/PPV/F1/FP/FN), timing-error (ms),
    RR-interval error (ms), and heart-rate error (bpm). Segments with no
    ground-truth peaks return n_true=0 and NaN rates (exclude from means, but
    COUNT them -- a reconstruction that yields peaks where truth has none is a
    real failure; see caller handling).
    """
    tol = int(round(tol_ms / 1000 * fs))
    t_pk = detect_r_peaks(true_ecg, fs, distance, prominence)
    p_pk = detect_r_peaks(pred_ecg, fs, distance, prominence)

    matches, TP, FP, FN, terr = match_peaks(t_pk, p_pk, tol)
    sens = TP / (TP + FN) if (TP + FN) else np.nan
    ppv  = TP / (TP + FP) if (TP + FP) else np.nan
    f1   = (2 * sens * ppv / (sens + ppv)
            if (sens and ppv and not np.isnan(sens) and not np.isnan(ppv)
                and (sens + ppv) > 0) else np.nan)

    terr_ms = np.abs(terr) / fs * 1000.0
    mae_t   = float(np.mean(terr_ms)) if terr_ms.size else np.nan
    med_t   = float(np.median(terr_ms)) if terr_ms.size else np.nan
    iqr_t   = float(np.percentile(terr_ms, 75) - np.percentile(terr_ms, 25)) \
              if terr_ms.size else np.nan

    # RR-interval error on consecutive MATCHED pairs only (avoids missed-beat bias)
    rr_err = np.nan
    if len(matches) >= 2:
        tp_sorted = sorted(matches)
        rr_true, rr_pred = [], []
        for (t0, p0), (t1, p1) in zip(tp_sorted[:-1], tp_sorted[1:]):
            rr_true.append((t1 - t0) / fs * 1000.0)
            rr_pred.append((p1 - p0) / fs * 1000.0)
        rr_err = float(np.mean(np.abs(np.array(rr_true) - np.array(rr_pred))))

    # Heart-rate error (bpm), estimator = 60000 / median-RR
    def _hr(pk):
        if len(pk) < 2:
            return np.nan
        return 60000.0 / np.median(np.diff(pk) / fs * 1000.0)
    hr_true, hr_pred = _hr(t_pk), _hr(p_pk)
    hr_err = float(abs(hr_true - hr_pred)) if not (np.isnan(hr_true) or np.isnan(hr_pred)) else np.nan

    return {
        "n_true": int(len(t_pk)), "n_pred": int(len(p_pk)),
        "TP": TP, "FP": FP, "FN": FN,
        "sensitivity": sens, "ppv": ppv, "f1": f1,
        "mae_timing_ms": mae_t, "median_timing_ms": med_t, "iqr_timing_ms": iqr_t,
        "rr_error_ms": rr_err,
        "hr_true_bpm": hr_true, "hr_pred_bpm": hr_pred, "hr_error_bpm": hr_err,
    }


def aggregate_temporal(rows):
    """Aggregate a list of per-segment temporal_rpeak_metrics dicts.

    Detection totals (TP/FP/FN) are POOLED then Se/PPV/F1 recomputed (the
    correct way to aggregate detection over segments). Timing/RR/HR errors are
    averaged over segments where they are defined; the valid-N is reported so
    the denominator is transparent.
    """
    rows = [r for r in rows if r is not None]
    TP = sum(r["TP"] for r in rows)
    FP = sum(r["FP"] for r in rows)
    FN = sum(r["FN"] for r in rows)
    sens = TP / (TP + FN) if (TP + FN) else np.nan
    ppv  = TP / (TP + FP) if (TP + FP) else np.nan
    f1   = 2 * sens * ppv / (sens + ppv) if (sens + ppv) else np.nan

    def _mean(key):
        v = [r[key] for r in rows if r[key] is not None and not np.isnan(r[key])]
        return (float(np.mean(v)), float(np.std(v)), len(v)) if v else (np.nan, np.nan, 0)

    out = {"n_segments": len(rows), "TP": TP, "FP": FP, "FN": FN,
           "sensitivity": sens, "ppv": ppv, "f1": f1}
    for k in ["mae_timing_ms", "median_timing_ms", "iqr_timing_ms",
              "rr_error_ms", "hr_error_bpm"]:
        m, s, n = _mean(k)
        out[k + "_mean"], out[k + "_std"], out[k + "_validN"] = m, s, n
    return out


# ---------------------------------------------------------------------------
# Preprocessing helpers (shared with the paper pipeline)
# ---------------------------------------------------------------------------
def ensure_positive_polarity(ecg):
    """Flip so R-peaks point up. Applied to the ECG TARGET only (never PCG)."""
    ecg = np.asarray(ecg)
    return -ecg if np.abs(ecg.min()) > np.abs(ecg.max()) else ecg


def normalize_zscore(sig):
    sig = np.asarray(sig, dtype=np.float32)
    return ((sig - sig.mean()) / (sig.std() + 1e-8)).astype(np.float32)

In [ ]:
%%writefile /content/ieee_style.py
"""
IEEE double-column figure style for the JBHI revision.

Design goals (per IEEE Transactions / J-BHI figure rules):
  * Vector output (PDF) + high-DPI PNG (600) for every figure.
  * Serif face (Times) to match IEEE body text; DejaVu Serif fallback on Colab.
  * BOLD axis labels that always carry explicit units.
  * Thick spines and ticks so lines stay legible at column scale.
  * constrained_layout everywhere -> no text/label/legend overlaps.
  * Column widths fixed to IEEE spec: single = 3.5 in, double = 7.16 in.

Usage
-----
    from ieee_style import set_ieee_style, new_fig, finalize_axes, save_fig
    set_ieee_style()
    fig, ax = new_fig("single", height=2.2)
    ax.plot(t, y)
    finalize_axes(ax, xlabel="Time (s)", ylabel="Amplitude (a.u.)")
    save_fig(fig, out_dir, "fig_name")   # writes fig_name.pdf AND fig_name.png
"""

import matplotlib
matplotlib.use("Agg")            # headless-safe; remove if you want inline display
import matplotlib.pyplot as plt

# IEEE column widths (inches)
COL_SINGLE = 3.5
COL_DOUBLE = 7.16

# Colour-blind-safe palette (Okabe-Ito), stable role -> colour mapping
COLORS = {
    "truth":    "#009E73",   # bluish green  -> ground truth
    "proposed": "#0072B2",   # blue          -> proposed model
    "cnn":      "#D55E00",   # vermillion    -> CNN
    "unet":     "#CC79A7",   # reddish purple-> U-Net
    "wgan":     "#E69F00",   # orange        -> WGAN-GP
    "pcg":      "#666666",   # grey          -> PCG input
    "accent":   "#0072B2",
    "muted":    "#999999",
}


def set_ieee_style():
    """Apply the global rcParams. Call once at the top of every script."""
    plt.rcParams.update({
        "figure.dpi":        150,
        "savefig.dpi":       600,
        "savefig.bbox":      "tight",
        "savefig.pad_inches": 0.02,
        "figure.constrained_layout.use": True,
        "font.size":         8,
        "font.family":       "serif",
        "font.serif":        ["Times New Roman", "Times", "DejaVu Serif"],
        "mathtext.fontset":  "dejavuserif",
        "text.usetex":       False,
        "axes.labelsize":    9,
        "axes.labelweight":  "bold",
        "axes.titlesize":    9,
        "axes.titleweight":  "bold",
        "axes.linewidth":    1.1,
        "axes.grid":         True,
        "grid.alpha":        0.25,
        "grid.linewidth":    0.4,
        "xtick.labelsize":   8,
        "ytick.labelsize":   8,
        "xtick.major.width": 1.0,
        "ytick.major.width": 1.0,
        "xtick.major.size":  3.5,
        "ytick.major.size":  3.5,
        "xtick.direction":   "out",
        "ytick.direction":   "out",
        "legend.fontsize":   7.5,
        "legend.framealpha": 0.92,
        "legend.edgecolor":  "0.7",
        "legend.handlelength": 1.6,
        "lines.linewidth":   1.0,
        "lines.solid_capstyle": "round",
    })


def new_fig(width="single", height=2.2, nrows=1, ncols=1, **kw):
    """Create a figure sized to an IEEE column.

    width : 'single' (3.5 in), 'double' (7.16 in), or a float in inches.
    height: figure height in inches.
    Returns (fig, ax) or (fig, axes) exactly like plt.subplots.
    """
    w = {"single": COL_SINGLE, "double": COL_DOUBLE}.get(width, width)
    fig, ax = plt.subplots(nrows, ncols, figsize=(w, height), **kw)
    return fig, ax


def finalize_axes(ax, xlabel=None, ylabel=None, title=None,
                  legend=False, legend_loc="best", legend_ncol=1):
    """Apply bold labels with units, thicken spines, tidy ticks.

    Always pass xlabel/ylabel WITH units, e.g. 'Time (ms)', 'PSNR (dB)'.
    """
    if xlabel:
        ax.set_xlabel(xlabel, fontweight="bold")
    if ylabel:
        ax.set_ylabel(ylabel, fontweight="bold")
    if title:
        ax.set_title(title, fontweight="bold")
    for spine in ax.spines.values():
        spine.set_linewidth(1.1)
    ax.tick_params(axis="both", which="major", width=1.0, labelsize=8)
    if legend:
        leg = ax.legend(loc=legend_loc, ncol=legend_ncol, frameon=True)
        leg.get_frame().set_linewidth(0.6)
    return ax


def save_fig(fig, out_directory, name):
    """Save both a vector PDF (for LaTeX) and a 600-dpi PNG (for preview)."""
    import os
    pdf = os.path.join(out_directory, f"{name}.pdf")
    png = os.path.join(out_directory, f"{name}.png")
    fig.savefig(pdf)
    fig.savefig(png)
    plt.close(fig)
    print(f"  saved  {pdf}")
    print(f"  saved  {png}")
    return pdf, png

In [ ]:
import importlib
import config_paths, models, metrics, ieee_style
for m in (config_paths, models, metrics, ieee_style):
    importlib.reload(m)
ieee_style.set_ieee_style()
print('shared modules ready | device =', config_paths.DEVICE)

## Experiment

# exp5 - Computational Complexity + Inference Latency (Reviewer 2, comment R2.2)

Reviewer 2 asked us to report the algorithm's computational complexity and to
justify the deployment cost. This script measures, for every paper model:

  * trainable parameter count (asserted against the VERIFIED local counts),
  * MACs / FLOPs per single 8 s (64,000-sample) forward pass (thop -> ptflops -> fvcore),
  * inference latency on CPU and (if present) GPU: >=10 warmup + >=50 timed passes,
    reported as mean +/- SD with a real-time factor (RTF = 8 s / latency),
  * weight memory (MB) and GPU peak activation memory (MB).

It emits a JSON of raw numbers plus a paste-ready IEEE LaTeX complexity table,
and one IEEE double-column figure. Only the generator runs at inference; the
discriminator is discarded after training and is reported for context only.

VERIFIED expected parameter counts (asserted, warn-on-mismatch):
  CNN 9,776,385 | U-Net 13,609,345 | WGAN-GP 13,609,345 | Proposed 19,315,715 | Disc 2,176,833
The submitted paper's "14.0M" for the Proposed generator is an error; the true
generator has 19.3M parameters.

In [ ]:
# --- Standard shared-module header (see SHARED GROUND TRUTH) ----------------
# When run as a stand-alone .py in Colab you must FIRST copy the local
# `shared/` folder (config_paths.py, models.py, metrics.py, ieee_style.py) to
#   /content/drive/MyDrive/JBHI_Revision/shared
# OR paste those four modules next to this script. build_notebooks.py writes
# them into /content automatically, so a built notebook needs no upload.
from google.colab import drive
drive.mount('/content/drive')

import sys
for _p in ('/content', '/content/drive/MyDrive/JBHI_Revision/shared'):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import importlib
import config_paths
import models
import metrics          # noqa: F401  (imported for pipeline parity / availability check)
import ieee_style
for _m in (config_paths, models, ieee_style):
    importlib.reload(_m)
ieee_style.set_ieee_style()
print('shared modules ready | device =', config_paths.DEVICE)

In [ ]:
# --- Install a FLOPs/MACs profiler if none is present -----------------------
# Order of preference: thop -> ptflops -> fvcore. We install thop (the smallest)
# and additionally try fvcore, which traces matmuls and therefore also counts
# the bottleneck self-attention that thop/ptflops hooks miss.
import subprocess

def _ensure(pkg):
    try:
        __import__(pkg)
        print(f'  {pkg}: already available')
        return True
    except Exception:
        print(f'  {pkg}: installing ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
        try:
            __import__(pkg)
            print(f'  {pkg}: installed')
            return True
        except Exception:
            print(f'  {pkg}: NOT available (continuing)')
            return False

_HAS_THOP    = _ensure('thop')
_HAS_FVCORE  = _ensure('fvcore')
try:
    import ptflops          # noqa: F401
    _HAS_PTFLOPS = True
except Exception:
    _HAS_PTFLOPS = False
print('profilers  | thop:', _HAS_THOP, '| ptflops:', _HAS_PTFLOPS, '| fvcore:', _HAS_FVCORE)

In [ ]:
# --- Imports + run configuration --------------------------------------------
import os
import gc
import json
import time
import copy
import warnings

import numpy as np
import torch

OUT_DIR   = config_paths.out_dir('exp5_complexity')
SEG_LEN   = config_paths.SEG_LEN          # 64000 samples
FS        = config_paths.FS               # 8000 Hz
SIG_SEC   = SEG_LEN / FS                  # 8.0 s of signal per segment
WARMUP    = 10                            # >= 10 warmup passes (spec)
ITERS     = 50                            # >= 50 timed passes (spec)

CPU_DEVICE = torch.device('cpu')
GPU_DEVICE = torch.device('cuda') if torch.cuda.is_available() else None

print(f'output dir : {OUT_DIR}')
print(f'segment    : {SEG_LEN} samples = {SIG_SEC:.1f} s @ {FS} Hz')
print(f'benchmark  : {WARMUP} warmup + {ITERS} timed passes per device')
print(f'GPU        : {"available (" + torch.cuda.get_device_name(0) + ")" if GPU_DEVICE else "NOT available; CPU only"}')

In [ ]:
# --- Model registry ---------------------------------------------------------
# Each entry: constructor, checkpoint key, forward signature, expected params,
# role. `is_proposed` -> forward(pcg, scenario); baselines -> forward(pcg);
# the discriminator -> forward(pcg, ecg, scenario) and is training-only.
EXPECTED = {
    'CNN':      9_776_385,
    'U-Net':    13_609_345,
    'WGAN-GP':  13_609_345,
    'Proposed': 19_315_715,
    'Discriminator': 2_176_833,
}

def _build_registry():
    reg = [
        dict(name='CNN',      ctor=models.CNNEncoderDecoder,      ckpt='CNN',
             sig='pcg',        role='generator'),
        dict(name='U-Net',    ctor=models.UNetGenerator,          ckpt='UNet',
             sig='pcg',        role='generator'),
        dict(name='WGAN-GP',  ctor=models.UNetGenerator,          ckpt='WGAN-GP',
             sig='pcg',        role='generator'),
        dict(name='Proposed', ctor=models.ImprovedGeneratorNoTanh, ckpt='Proposed',
             sig='pcg+scen',   role='generator'),
        dict(name='Discriminator', ctor=models.SpectralNormDiscriminator, ckpt=None,
             sig='disc',       role='discriminator'),
    ]
    return reg

REGISTRY   = _build_registry()
GEN_ORDER  = ['CNN', 'U-Net', 'WGAN-GP', 'Proposed']   # inference-cost table order


def make_inputs(sig, device):
    """Return the positional-argument tuple this forward signature expects."""
    pcg  = torch.randn(1, 1, SEG_LEN, device=device)
    scen = torch.tensor([config_paths.SCEN_EXERCISE], dtype=torch.float32, device=device)
    if sig == 'pcg':
        return (pcg,)
    if sig == 'pcg+scen':
        return (pcg, scen)
    if sig == 'disc':
        ecg = torch.randn(1, 1, SEG_LEN, device=device)
        return (pcg, ecg, scen)
    raise ValueError(f'unknown signature {sig}')


def build_model(entry, device):
    """Instantiate and (best-effort) load the checkpoint. Complexity and latency
    depend only on the architecture, so a missing checkpoint is non-fatal:
    the fresh-initialised model is used and a note is printed."""
    model = entry['ctor']()
    loaded = False
    ckpt_key = entry['ckpt']
    if ckpt_key is not None:
        path = config_paths.CKPT.get(ckpt_key)
        if path and os.path.exists(path):
            try:
                model, missing = models.load_model(model, path, device=str(device))
                loaded = True
                print(f"  [{entry['name']}] loaded checkpoint ({os.path.basename(path)})")
            except Exception as e:
                print(f"  [{entry['name']}] checkpoint load FAILED ({e}); using fresh weights")
        else:
            print(f"  [{entry['name']}] checkpoint not found; using fresh weights (counts unaffected)")
    model.to(device).eval()
    return model, loaded

In [ ]:
# --- (1) Parameter counts, asserted against the VERIFIED numbers ------------
print('\n=== Parameter counts ===')
param_results = {}
for entry in REGISTRY:
    name = entry['name']
    model, loaded = build_model(entry, CPU_DEVICE)
    n = models.count_params(model)
    exp = EXPECTED[name]
    ok = (n == exp)
    flag = 'OK' if ok else 'MISMATCH'
    if not ok:
        warnings.warn(f'{name}: param count {n:,} != expected {exp:,}')
    print(f'  {name:<14} params = {n:>12,}   expected = {exp:>12,}   [{flag}]')
    param_results[name] = dict(
        params=int(n), expected=int(exp), matches_expected=bool(ok),
        weights_mb=float(n * 4 / 1e6),      # float32 weights, decimal MB
        checkpoint_loaded=bool(loaded), role=entry['role'],
    )
    del model
    gc.collect()

In [ ]:
# --- (2) MACs / FLOPs per 8 s forward pass ----------------------------------
# thop and ptflops register hooks on known layer types and therefore miss the
# raw matmuls inside the multi-head self-attention bottleneck. fvcore traces the
# graph and does count them, so where fvcore is available we report it too and
# treat it as the more complete estimate for the attention-bearing Proposed model.

def macs_thop(model, inputs):
    from thop import profile as thop_profile
    m = copy.deepcopy(model)                 # thop injects buffers; keep original clean
    macs, _ = thop_profile(m, inputs=inputs, verbose=False)
    del m
    return float(macs)

def macs_ptflops(model, inputs, sig):
    from ptflops import get_model_complexity_info
    m = copy.deepcopy(model)
    # ptflops feeds a single constructed input; map our positional args to kwargs.
    if sig == 'pcg':
        kw = dict(x=inputs[0])
    elif sig == 'pcg+scen':
        kw = dict(x=inputs[0], scenario_onehot=inputs[1])
    elif sig == 'disc':
        kw = dict(pcg=inputs[0], ecg=inputs[1], scenario_onehot=inputs[2])
    else:
        return float('nan')
    def _ctor(_res):
        return {k: v.clone() for k, v in kw.items()}
    macs, _ = get_model_complexity_info(
        m, (1, SEG_LEN), input_constructor=_ctor,
        as_strings=False, print_per_layer_stat=False, verbose=False)
    del m
    return float(macs)

def macs_fvcore(model, inputs):
    from fvcore.nn import FlopCountAnalysis
    m = copy.deepcopy(model)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        fca = FlopCountAnalysis(m, inputs)
        fca.unsupported_ops_warnings(False)
        fca.uncalled_modules_warnings(False)
        val = float(fca.total())            # fvcore counts one MAC as one "flop"
    del m
    return val

print('\n=== MACs / FLOPs per 8 s forward pass ===')
mac_results = {}
for entry in REGISTRY:
    name = entry['name']
    model, _ = build_model(entry, CPU_DEVICE)
    inputs = make_inputs(entry['sig'], CPU_DEVICE)
    per_backend = {}
    if _HAS_THOP:
        try:
            per_backend['thop'] = macs_thop(model, inputs)
        except Exception as e:
            print(f'    [{name}] thop failed: {e}')
    if _HAS_PTFLOPS:
        try:
            per_backend['ptflops'] = macs_ptflops(model, inputs, entry['sig'])
        except Exception as e:
            print(f'    [{name}] ptflops failed: {e}')
    if _HAS_FVCORE:
        try:
            per_backend['fvcore'] = macs_fvcore(model, inputs)
        except Exception as e:
            print(f'    [{name}] fvcore failed: {e}')

    # Headline MACs: prefer fvcore (counts attention) for attention models,
    # else thop, else ptflops, else NaN.
    headline_backend = None
    for cand in (('fvcore', 'thop', 'ptflops') if name == 'Proposed'
                 else ('thop', 'ptflops', 'fvcore')):
        if cand in per_backend and np.isfinite(per_backend[cand]):
            headline_backend = cand
            break
    macs = per_backend.get(headline_backend, float('nan')) if headline_backend else float('nan')
    gmac = macs / 1e9 if np.isfinite(macs) else float('nan')
    gflop = 2 * gmac if np.isfinite(gmac) else float('nan')
    print(f'  {name:<14} MACs = {gmac:8.2f} GMac  FLOPs = {gflop:8.2f} GFLOP  '
          f'(backend: {headline_backend})  all: '
          + ', '.join(f'{k}={v/1e9:.2f}G' for k, v in per_backend.items()))
    mac_results[name] = dict(
        macs=float(macs), gmac=float(gmac), gflop=float(gflop),
        backend=headline_backend,
        per_backend_gmac={k: float(v / 1e9) for k, v in per_backend.items()},
    )
    del model, inputs
    gc.collect()

In [ ]:
# --- (3) Inference latency: CPU and (if present) GPU ------------------------
# Generators only. RTF = signal_seconds / latency_seconds (RTF > 1 => faster
# than real-time). Peak GPU activation memory is read from the CUDA allocator.

def benchmark(model, inputs, device, warmup=WARMUP, iters=ITERS):
    model.to(device).eval()
    inputs = tuple(t.to(device) for t in inputs)
    times_ms = []
    is_cuda = (device.type == 'cuda')
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(*inputs)
        if is_cuda:
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats(device)
        for _ in range(iters):
            if is_cuda:
                ev0 = torch.cuda.Event(enable_timing=True)
                ev1 = torch.cuda.Event(enable_timing=True)
                ev0.record()
                _ = model(*inputs)
                ev1.record()
                torch.cuda.synchronize()
                times_ms.append(ev0.elapsed_time(ev1))
            else:
                t0 = time.perf_counter()
                _ = model(*inputs)
                t1 = time.perf_counter()
                times_ms.append((t1 - t0) * 1e3)
    times_ms = np.asarray(times_ms, dtype=float)
    mean_ms = float(times_ms.mean())
    std_ms  = float(times_ms.std())
    rtf     = float(SIG_SEC / (mean_ms / 1e3)) if mean_ms > 0 else float('nan')
    peak_mb = float(torch.cuda.max_memory_allocated(device) / 1e6) if is_cuda else float('nan')
    return dict(mean_ms=mean_ms, std_ms=std_ms, rtf=rtf, peak_mem_mb=peak_mb,
                n_timed=int(iters), n_warmup=int(warmup))

print('\n=== Inference latency (generators) ===')
lat_results = {}
for name in GEN_ORDER:
    entry = next(e for e in REGISTRY if e['name'] == name)
    rec = {}

    # CPU
    model, _ = build_model(entry, CPU_DEVICE)
    inputs = make_inputs(entry['sig'], CPU_DEVICE)
    rec['cpu'] = benchmark(model, inputs, CPU_DEVICE)
    print(f'  {name:<14} CPU  {rec["cpu"]["mean_ms"]:9.1f} +/- {rec["cpu"]["std_ms"]:6.1f} ms   '
          f'RTF {rec["cpu"]["rtf"]:6.2f}x')
    del model, inputs
    gc.collect()

    # GPU (optional)
    if GPU_DEVICE is not None:
        model, _ = build_model(entry, GPU_DEVICE)
        inputs = make_inputs(entry['sig'], GPU_DEVICE)
        rec['gpu'] = benchmark(model, inputs, GPU_DEVICE)
        print(f'  {name:<14} GPU  {rec["gpu"]["mean_ms"]:9.2f} +/- {rec["gpu"]["std_ms"]:6.2f} ms   '
              f'RTF {rec["gpu"]["rtf"]:8.1f}x   peak {rec["gpu"]["peak_mem_mb"]:7.1f} MB')
        del model, inputs
        torch.cuda.empty_cache()
        gc.collect()
    else:
        rec['gpu'] = None
    lat_results[name] = rec

In [ ]:
# --- (4) Assemble + save the JSON of raw numbers ----------------------------
results = dict(
    meta=dict(
        experiment='exp5_complexity_latency',
        reviewer_comment='R2.2',
        segment_samples=int(SEG_LEN), fs_hz=int(FS), signal_seconds=float(SIG_SEC),
        warmup=int(WARMUP), timed_iters=int(ITERS),
        gpu_name=(torch.cuda.get_device_name(0) if GPU_DEVICE is not None else None),
        cpu_only=bool(GPU_DEVICE is None),
        torch_version=str(torch.__version__),
        profilers=dict(thop=_HAS_THOP, ptflops=_HAS_PTFLOPS, fvcore=_HAS_FVCORE),
        note=('Only the generator runs at inference; the discriminator is '
              'discarded after training. thop and ptflops do not count the '
              'bottleneck self-attention matmuls, so fvcore is preferred for '
              'the Proposed model where available.'),
    ),
    params=param_results,
    macs=mac_results,
    latency=lat_results,
)
json_path = os.path.join(OUT_DIR, 'exp5_results.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)
print(f'\nsaved  {json_path}')

In [ ]:
# --- (5) Paste-ready IEEE LaTeX complexity table ----------------------------
# Values are filled from the measured numbers above. Prose in the caption obeys
# the manuscript writing rules (active voice, thin space before units, no em dash).

def _fmt(v, nd=1):
    return ('%.*f' % (nd, v)) if (v is not None and np.isfinite(v)) else '--'

def _lat_cell(rec):
    if rec is None:
        return '--'
    return r'%s\,$\pm$\,%s' % (_fmt(rec['mean_ms'], 1), _fmt(rec['std_ms'], 1))

gpu_present = GPU_DEVICE is not None
lines = []
lines.append('% ---------------------------------------------------------------------------')
lines.append('% exp5 complexity + latency table (auto-generated from exp5_results.json)')
lines.append('% Requires \\usepackage{booktabs}. Paste into Section III (Method).')
lines.append('% ---------------------------------------------------------------------------')
lines.append(r'\begin{table}[!t]')
lines.append(r'\centering')
lines.append(r'\caption{Computational complexity and inference latency per 8\,s segment '
             r'(64{,}000 samples). Only the generator runs at inference; the discriminator '
             r'is discarded after training and is listed for context. Latency is the mean '
             r'$\pm$\,SD over %d timed passes after %d warmup passes.}' % (ITERS, WARMUP))
lines.append(r'\label{tab:complexity}')
lines.append(r'\setlength{\tabcolsep}{4pt}')
lines.append(r'\begin{tabular}{lrrrrr}')
lines.append(r'\toprule')
lines.append(r'\textbf{Model} & \textbf{Params (M)} & \textbf{MACs (GMac)} & '
             r'\textbf{CPU (ms)} & \textbf{GPU (ms)} & \textbf{RTF} \\')
lines.append(r'\midrule')

for name in GEN_ORDER:
    p   = param_results[name]
    m   = mac_results[name]
    lat = lat_results[name]
    cpu_cell = _lat_cell(lat['cpu'])
    gpu_cell = _lat_cell(lat['gpu']) if gpu_present else '--'
    # RTF: report the deployment device (GPU when present, else CPU).
    if gpu_present and lat['gpu'] is not None:
        rtf_cell = r'%s$\times$' % _fmt(lat['gpu']['rtf'], 1)
    else:
        rtf_cell = r'%s$\times$' % _fmt(lat['cpu']['rtf'], 1)
    bold = (name == 'Proposed')
    row = r'%s & %s & %s & %s & %s & %s \\' % (
        (r'\textbf{%s}' % name) if bold else name,
        _fmt(p['params'] / 1e6, 2),
        _fmt(m['gmac'], 1),
        cpu_cell, gpu_cell, rtf_cell)
    lines.append(row)

lines.append(r'\midrule')
dp = param_results['Discriminator']
dm = mac_results['Discriminator']
lines.append(r'Discriminator$^{\dagger}$ & %s & %s & -- & -- & -- \\' % (
    _fmt(dp['params'] / 1e6, 2), _fmt(dm['gmac'], 1)))
lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')
lines.append(r'\vspace{2pt}')
lines.append(r'\\[-2pt]{\footnotesize $^{\dagger}$Training only; not part of the deployed model.}')
lines.append(r'\end{table}')

latex_table = '\n'.join(lines)
table_path = os.path.join(OUT_DIR, 'exp5_complexity_table.tex')
with open(table_path, 'w', encoding='utf-8') as f:
    f.write(latex_table + '\n')
print(f'saved  {table_path}\n')
print(latex_table)

In [ ]:
# --- (6) IEEE double-column figure: parameters, MACs, and latency -----------
import matplotlib.pyplot as plt

CMAP = {
    'CNN':      ieee_style.COLORS['cnn'],
    'U-Net':    ieee_style.COLORS['unet'],
    'WGAN-GP':  ieee_style.COLORS['wgan'],
    'Proposed': ieee_style.COLORS['proposed'],
}
names   = GEN_ORDER
xpos    = np.arange(len(names))
params_m = [param_results[n]['params'] / 1e6 for n in names]
gmacs    = [mac_results[n]['gmac'] for n in names]
cpu_ms   = [lat_results[n]['cpu']['mean_ms'] for n in names]
cpu_sd   = [lat_results[n]['cpu']['std_ms'] for n in names]
if gpu_present:
    gpu_ms = [lat_results[n]['gpu']['mean_ms'] for n in names]
    gpu_sd = [lat_results[n]['gpu']['std_ms'] for n in names]

fig, axes = ieee_style.new_fig('double', height=2.7, ncols=3)
ax0, ax1, ax2 = axes

# Panel (a): parameters (M)
bars0 = ax0.bar(xpos, params_m, color=[CMAP[n] for n in names],
                width=0.68, edgecolor='black', linewidth=0.5)
ax0.set_xticks(xpos)
ax0.set_xticklabels(names, rotation=30, ha='right')
ieee_style.finalize_axes(ax0, xlabel='Model', ylabel='Parameters (M)')

# Panel (b): MACs (GMac)
ax1.bar(xpos, gmacs, color=[CMAP[n] for n in names],
        width=0.68, edgecolor='black', linewidth=0.5)
ax1.set_xticks(xpos)
ax1.set_xticklabels(names, rotation=30, ha='right')
ieee_style.finalize_axes(ax1, xlabel='Model', ylabel='MACs (GMac)')

# Panel (c): latency, CPU vs GPU (log scale, ms)
w = 0.38
ax2.bar(xpos - (w/2 if gpu_present else 0), cpu_ms, width=w, yerr=cpu_sd,
        capsize=2, color=ieee_style.COLORS['muted'], edgecolor='black',
        linewidth=0.5, label='CPU')
if gpu_present:
    ax2.bar(xpos + w/2, gpu_ms, width=w, yerr=gpu_sd, capsize=2,
            color=ieee_style.COLORS['accent'], edgecolor='black',
            linewidth=0.5, label='GPU')
ax2.set_yscale('log')
ax2.set_xticks(xpos)
ax2.set_xticklabels(names, rotation=30, ha='right')
ieee_style.finalize_axes(ax2, xlabel='Model', ylabel='Latency (ms)',
                         legend=True, legend_loc='upper left')

ieee_style.save_fig(fig, OUT_DIR, 'exp5_complexity_latency')

In [ ]:
# --- (7) Console summary ----------------------------------------------------
print('\n=== exp5 summary ===')
print(f'{"Model":<14}{"Params(M)":>11}{"GMac":>9}{"CPU ms":>12}{"GPU ms":>14}{"RTF":>10}')
for name in GEN_ORDER:
    p = param_results[name]; m = mac_results[name]; lat = lat_results[name]
    cpu = lat['cpu']; gpu = lat['gpu']
    cpu_s = f'{cpu["mean_ms"]:.1f}+/-{cpu["std_ms"]:.1f}'
    gpu_s = (f'{gpu["mean_ms"]:.2f}+/-{gpu["std_ms"]:.2f}' if gpu else 'n/a')
    rtf   = (gpu['rtf'] if gpu else cpu['rtf'])
    print(f'{name:<14}{p["params"]/1e6:>11.2f}{m["gmac"]:>9.1f}{cpu_s:>12}{gpu_s:>14}{rtf:>9.1f}x')
d = param_results['Discriminator']
print(f'{"Disc (train)":<14}{d["params"]/1e6:>11.2f}{mac_results["Discriminator"]["gmac"]:>9.1f}'
      f'{"--":>12}{"--":>14}{"--":>10}')
print('\nOnly the generator runs at inference; the discriminator is discarded after training.')
print(f'Artifacts in: {OUT_DIR}')
print('  - exp5_results.json')
print('  - exp5_complexity_table.tex')
print('  - exp5_complexity_latency.pdf / .png')